# Text-to-Speech and Speech-to-Text on Scripture Text

This notebook covers:

1. **2 sophisticated models**, both loaded through the `transformers` library:
   - MMS-TTS (Meta) — text-to-speech
   - Whisper (OpenAI) — speech-to-text
2. **2 languages**: English and Hindi
3. **4 performance metrics**: WER, CER, BLEU, RTF
4. **Corpus**: a real, referenced scripture corpus — the King James Bible (English,
   public domain) and the Bhagavad Gita (Hindi, translated here from the public-domain
   Sanskrit original)

Pipeline tested: scripture text → TTS → audio → STT → transcribed text, then
transcribed text is scored against the original.

Run all cells in order.


## 0. Setup

Install the required libraries. First run takes 2–3 minutes.

In [ ]:
!pip install -q transformers accelerate torch torchaudio
!pip install -q soundfile librosa
!pip install -q jiwer sacrebleu


In [ ]:
import torch
import time
import numpy as np
import librosa
import soundfile as sf
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 1. Corpus: Scripture Text

**English** — 12 verses from the King James Bible (public domain, 1611), spanning
several books for lexical variety.

**Hindi** — 12 verses from the Bhagavad Gita. The Sanskrit original is ancient and
public domain; the Hindi text below is translated here directly from that Sanskrit,
rather than copied from any single published translator's copyrighted wording.

Each line carries its source reference, so results can be traced back to a specific
verse.


In [ ]:
corpus = {
    "eng": [
        ("In the beginning God created the heaven and the earth.", "Genesis 1:1"),
        ("And God said, Let there be light: and there was light.", "Genesis 1:3"),
        ("Thou shalt have no other gods before me.", "Exodus 20:3"),
        ("The Lord is my shepherd; I shall not want.", "Psalm 23:1"),
        ("God is our refuge and strength, a very present help in trouble.", "Psalm 46:1"),
        ("The fear of the Lord is the beginning of knowledge.", "Proverbs 1:7"),
        ("To every thing there is a season, and a time to every purpose under the heaven.", "Ecclesiastes 3:1"),
        ("But they that wait upon the Lord shall renew their strength.", "Isaiah 40:31"),
        ("In the beginning was the Word, and the Word was with God, and the Word was God.", "John 1:1"),
        ("For God so loved the world, that he gave his only begotten Son.", "John 3:16"),
        ("Blessed are the peacemakers: for they shall be called the children of God.", "Matthew 5:9"),
        ("And we know that all things work together for good to them that love God.", "Romans 8:28"),
    ],
    "hin": [
        ("आत्मा न कभी जन्म लेती है, न कभी मरती है।", "Gita 2.20"),
        ("हे कुंतीपुत्र, सर्दी-गर्मी और सुख-दुःख इंद्रियों के विषयों से उत्पन्न होते हैं।", "Gita 2.14"),
        ("तुम्हारा अधिकार केवल कर्म करने में है, उसके फल में कभी नहीं।", "Gita 2.47"),
        ("हे अर्जुन, आसक्ति को त्यागकर और योग में स्थित होकर कर्म करो।", "Gita 2.48"),
        ("समबुद्धि वाला व्यक्ति इस जीवन में पाप और पुण्य दोनों को त्याग देता है।", "Gita 2.50"),
        ("अपना धर्म, भले ही अपूर्ण हो, दूसरे के धर्म से श्रेष्ठ है।", "Gita 3.35"),
        ("हे भारत, जब-जब धर्म की हानि होती है, तब-तब मैं प्रकट होता हूँ।", "Gita 4.7"),
        ("साधुओं की रक्षा और दुष्टों के विनाश के लिए मैं युग-युग में जन्म लेता हूँ।", "Gita 4.8"),
        ("मनुष्य को स्वयं अपने बल से अपना उद्धार करना चाहिए, स्वयं को कभी गिरने नहीं देना चाहिए।", "Gita 6.5"),
        ("अभ्यास और वैराग्य से मन को वश में किया जा सकता है।", "Gita 6.35"),
        ("क्रोध से मोह उत्पन्न होता है, और मोह से बुद्धि नष्ट हो जाती है।", "Gita 2.63"),
        ("सभी कर्तव्यों को छोड़कर केवल मेरी शरण में आ जाओ।", "Gita 18.66"),
    ],
}

for lang, items in corpus.items():
    print(f"[{lang}] {len(items)} lines")
    for text, ref in items[:3]:
        print(f"  ({ref}) {text}")
    print("  ...\n")


## 2. Model 1 — Text-to-Speech: MMS-TTS

Meta's Massively Multilingual Speech (MMS) project ships one TTS checkpoint per
language — a VITS-based neural model (flow-based, end-to-end text-to-waveform,
no separate vocoder step). We load the native English and Hindi checkpoints.


In [ ]:
from transformers import VitsModel, AutoTokenizer

TTS_CHECKPOINTS = {
    "eng": "facebook/mms-tts-eng",
    "hin": "facebook/mms-tts-hin",
}

tts_models = {}
for lang, name in TTS_CHECKPOINTS.items():
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = VitsModel.from_pretrained(name).to(device).eval()
    tts_models[lang] = (tokenizer, model)

print("TTS checkpoints loaded:", list(tts_models.keys()))


In [ ]:
def synthesize(text, lang):
    tokenizer, model = tts_models[lang]
    inputs = tokenizer(text, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        waveform = model(**inputs).waveform
    elapsed = time.time() - start

    audio = waveform.squeeze().cpu().numpy()
    sr = model.config.sampling_rate
    return audio, sr, elapsed

# Quick listen: first line per language
for lang, items in corpus.items():
    text, ref = items[0]
    audio, sr, elapsed = synthesize(text, lang)
    print(f"[{lang}] ({ref}) synthesized {len(audio)/sr:.2f}s of audio in {elapsed:.2f}s")
    display(Audio(audio, rate=sr))


## 3. Model 2 — Speech-to-Text: Whisper

OpenAI's Whisper is a single multilingual encoder-decoder transformer covering ~99
languages, including English and Hindi, in one checkpoint (unlike MMS-TTS, which
needs one checkpoint per language).


In [ ]:
from transformers import pipeline

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device=0 if device == "cuda" else -1,
)

WHISPER_LANGUAGE = {"eng": "english", "hin": "hindi"}

def transcribe(audio, sr, lang):
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        sr = 16000
    result = asr_pipe(
        {"array": audio, "sampling_rate": sr},
        generate_kwargs={"language": WHISPER_LANGUAGE[lang]},
    )
    return result["text"].strip()

# Quick check on the same two lines used above
for lang, items in corpus.items():
    text, ref = items[0]
    audio, sr, _ = synthesize(text, lang)
    transcribed = transcribe(audio, sr, lang)
    print(f"[{lang}] ({ref}) original:    {text}")
    print(f"[{lang}] ({ref}) transcribed: {transcribed}\n")


## 4. Full Round Trip: Text → TTS → Audio → STT → Text

Every verse, in both languages, goes through the full pipeline. Timing is kept for
the RTF metric below. With 24 lines total, this cell takes a few minutes on CPU —
faster with a GPU runtime (Runtime → Change runtime type → GPU).


In [ ]:
records = []

for lang, items in corpus.items():
    for text, ref in items:
        audio, sr, tts_time = synthesize(text, lang)
        duration = len(audio) / sr
        transcribed = transcribe(audio, sr, lang)
        records.append({
            "lang": lang,
            "ref": ref,
            "original": text,
            "transcribed": transcribed,
            "tts_time": tts_time,
            "audio_duration": duration,
        })

print(f"Processed {len(records)} lines.")
for r in records[:4]:
    print(f"[{r['lang']}] ({r['ref']}) original:    {r['original']}")
    print(f"[{r['lang']}] ({r['ref']}) transcribed: {r['transcribed']}\n")
print("...")


In [ ]:
import pandas as pd

records_df = pd.DataFrame(records)[["lang", "ref", "original", "transcribed"]]
records_df


## 5. Performance Metrics

| Metric | Measures | Range |
|---|---|---|
| **WER** | word-level edit distance, original vs. transcribed | 0+, lower is better |
| **CER** | character-level edit distance | 0+, lower is better |
| **BLEU** | n-gram overlap, original vs. transcribed | 0–100, higher is better |
| **RTF** | TTS synthesis time ÷ audio duration | lower is faster than real-time |

WER/CER/BLEU measure round-trip intelligibility: how well the transcribed text
recovers the original. RTF measures TTS speed, independent of accuracy. Computed
per language, over all 12 verses each — enough lines for the numbers to mean
something, unlike a 2–3 sentence spot check.


In [ ]:
import jiwer
import sacrebleu

def compute_metrics(records, lang):
    subset = [r for r in records if r["lang"] == lang]
    refs = [r["original"] for r in subset]
    hyps = [r["transcribed"] for r in subset]

    wer = jiwer.wer(refs, hyps)
    cer = jiwer.cer(refs, hyps)
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    rtf = sum(r["tts_time"] for r in subset) / sum(r["audio_duration"] for r in subset)

    return {
        "WER": round(wer, 4),
        "CER": round(cer, 4),
        "BLEU": round(bleu, 2),
        "RTF": round(rtf, 3),
        "n_lines": len(subset),
    }

results = {lang: compute_metrics(records, lang) for lang in corpus}
metrics_df = pd.DataFrame(results).T
metrics_df.index.name = "Language"
metrics_df


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

metrics_df[["WER", "CER"]].plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_ylabel("Error rate (lower is better)")
axes[0].set_title("WER and CER by language")

metrics_df[["BLEU"]].plot(kind="bar", ax=axes[1], rot=0, legend=False, color="#55a868")
axes[1].set_ylabel("BLEU (higher is better)")
axes[1].set_title("BLEU by language")

plt.tight_layout()
plt.show()


## Conclusion

This notebook demonstrated:

- **2 sophisticated models** via `transformers`: MMS-TTS (text-to-speech) and Whisper
  (speech-to-text)
- **2 languages**: English and Hindi, both natively supported by each model
- **4 metrics**: WER, CER, BLEU (round-trip intelligibility) and RTF (synthesis speed)
- **Corpus**: 12 King James Bible verses (English) and 12 Bhagavad Gita verses
  (Hindi, translated from the public-domain Sanskrit), each traceable to its source
  reference

Compare WER/CER/BLEU across the two languages and scriptures to see whether either
model favors one over the other, and compare RTF to see which language synthesizes
faster.
